# 05 — Full Pipeline

End-to-end run: set a bounding box and get speed limit estimates for every
Overture segment in the area, evaluated against Overture's own normalized
speed limit values.

**Usage:**
1. Set `BBOX` and `TOKEN`
2. Run all cells
3. Find the final GeoParquet in `speed_limit_estimates.parquet`

In [ ]:
import os
import geopandas as gpd
from slc import consensus, evaluate, fetch, match, snap, split, viz

TOKEN = os.environ['MAPILLARY_ACCESS_TOKEN']

# ── Configuration ──────────────────────────────────────────────────────────
BBOX = (-111.920, 40.855, -111.855, 40.910)  # Bountiful, UT

SNAP_MAX_DIST_M      = 30.0
SNAP_MAX_HDG_DIFF    = 30.0
SPLIT_BEARING_THRESH = 60.0
SPLIT_WINDOW_M       = 80.0
MATCH_MAX_DIST_M     = 25.0
MATCH_MIN_OVERLAP    = 0.3
# ───────────────────────────────────────────────────────────────────────────

In [ ]:
print('1/5  Fetching Mapillary data...')
signs     = fetch.fetch_mapillary_signs(BBOX, TOKEN)
images    = fetch.fetch_mapillary_images(BBOX, TOKEN)
sequences = fetch.build_sequences(images)
print(f'     Signs: {len(signs)}  |  Sequences: {len(sequences)}')

print('2/5  Fetching Overture segments (includes normalized speed limits)...')
overture_raw = fetch.fetch_overture_segments(BBOX)
overture     = fetch.extract_overture_speed_limits(overture_raw)
print(f'     Segments: {len(overture)}  |  With speed limit: {overture["speed_limit_value"].notna().sum()}')

In [ ]:
print('3/5  Snapping signs and splitting sequences...')
snapped     = snap.snap_signs_to_sequences(signs, sequences,
                  max_distance_m=SNAP_MAX_DIST_M,
                  max_heading_diff=SNAP_MAX_HDG_DIFF)
split_edges = split.split_all_sequences(sequences, snapped,
                  bearing_threshold_deg=SPLIT_BEARING_THRESH,
                  window_m=SPLIT_WINDOW_M)
labeled = split_edges['speed_mph'].notna().sum()
print(f'     Labeled edges: {labeled}/{len(split_edges)}')

In [ ]:
print('4/5  Matching to Overture segments...')
matches   = match.match_edges_to_overture(split_edges, overture,
                max_distance_m=MATCH_MAX_DIST_M,
                min_overlap_fraction=MATCH_MIN_OVERLAP)
estimates = consensus.compute_consensus(matches)
print(f'     Estimates: {len(estimates)}')

In [ ]:
print('5/5  Evaluating against Overture speed limits...')
comparison = evaluate.compare_to_overture(estimates, overture)
metrics    = evaluate.compute_metrics(comparison)
report     = evaluate.generate_report(comparison, metrics)
print(report)

In [ ]:
# Export final dataset
estimates.to_parquet('speed_limit_estimates.parquet', index=False)
print('Saved speed_limit_estimates.parquet')

print(f"\nSummary:")
print(f"  Segments with estimates : {len(estimates)}")
print(f"  Exact match vs Overture : {metrics.get('exact_match_rate', 'N/A')}")
print(f"  Within 5 mph vs Overture: {metrics.get('within_5mph_rate', 'N/A')}")

In [ ]:
# Final map
viz.map_comparison(comparison, overture)